In [1]:
import pandas as pd
import sys
from jira import JIRA
jira=JIRA('https://research.quantumgalaxy.cn/', basic_auth=('bot2', "jira_bot2"))

In [2]:
def iter_jira_search(jql,maxResults=500,startAt=0):
    maxResults=maxResults
    startAt=startAt
    result=[]
    while True:
        result1=jira.search_issues(jql,maxResults=maxResults,startAt=startAt)
        
        if result1.current+len(result)>=result1.total:
            print('break')
            print('total issue:',len(result))
            break
        else:
            result+=result1
            print(len(result))
            #print(result1.total)
            #print(len(result)>=result1.total)
            
            startAt+=len(result1)
            #print(startAt)
    return result


In [73]:
issues=iter_jira_search('project = COMPSTUDY AND issuetype = 公司研究',maxResults=500)

500


In [1]:
issues

NameError: name 'issues' is not defined

In [36]:
target_jira.isLast

0

In [26]:
for issue in target_jira:
    if issue.fields.__dict__.get('customfield_11216'):
        if  (issue.fields.__dict__.get('customfield_10201')):
            print(issue.fields.summary)
            print(issue.fields.customfield_10201)
            

In [25]:
issue1.fields.summary

'奥特维'

In [20]:
issue.fields.__dict__.get('customfield_11216')

In [21]:
issue.fields.customfield_11216

In [3]:
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.neoapi import Neo4j
uri = "neo4j+ssc://534ea9b7.databases.neo4j.io:7687"
user = "neo4j"
password = "QuantumGalaxy"
neo=Neo4j(uri,user,password)
codes=neo.fetch_company_and_indicator_with_code()

In [4]:
import re
from tqdm import tqdm

In [10]:
code='002229.SZ'
issues=jira.search_issues(f'project = COMPSTUDY AND issuetype = 公司研究 AND 万得代码 ={code}')

In [11]:
issues

[<JIRA Issue: key='COMPSTUDY-6388', id='29443'>]

In [14]:
clue=issue.fields.__dict__.get('customfield_11216')
if clue:
    print(clue)

赛微电子7月16日公告，近日，公司控股子公司以MEMS工艺为某客户制造的系列BAW滤波器完成了小批量试生产阶段，开始进行BAW滤波器规模化量产。慈星股份参股武汉敏声，武汉敏声委托的塞微电子生产BAW滤波器（供应华为5G）。


In [4]:
with open('doc.txt','r') as r:
    doc=r.read()
items=doc.split('\n')


FileNotFoundError: [Errno 2] No such file or directory: 'doc.txt'

In [97]:
item=items[0]
item

'- 昊志机电300503.SZ(60.82亿元,+19.99%)：公司在空心杯电机，无框力矩电机，力传感器，谐波减速器方面均有产品，国内目前只有两家公司能做无框力矩电机，一家是步科股份，另一家为昊志机电；公司掌握六维力矩传感器技术，一个六维力矩传感器的价值量是2万元；减速器年产能为15万台。'

In [98]:
pattern = r'^(- )(.*?)\s*([A-Z0-9.]+)\s*\(([\d.]+[亿元万]*),([+-]?\d+\.\d+%)\)：(.*)$'
pattern=re.compile(r'^(- *)(.*?)\s*([A-Z0-9.]+)\s*\(([\d.]+[亿元万]*),([+-]?\d+\.\d+%)\)：(.*)$')
comps=[]
for item in items:
    matches = pattern.match(item)

    if matches:
        start = matches.group(1)
        name = matches.group(2)
        code = matches.group(3)
        market_value = matches.group(4)
        increase = matches.group(5)
        intro = matches.group(6)
        comps.append((name,code,intro))
    else:print(item)

In [99]:
comps[-1]

('苏常柴Ａ',
 '000570.SZ',
 '公司6月28日公告称，已收到常州市钟楼区给予的征收子公司房屋补偿款7000万元，该笔收入将计入中报。以此预计，苏常柴A中报业绩或仍有望翻倍。')

In [106]:
issue_list=[]

with tqdm(comps,desc='here') as t:
    for name, code, intro in t:
        
        get_issue=jira.search_issues(f'project = COMPSTUDY AND 万得代码 ={code}')
        if get_issue:
            issue=get_issue[0]
            try:
                issue.update(description='',customfield_11216=intro)
            except:print('error on :',name,code)
            #zhencha:customfield_11216
            #print('update exist issue:',name)
            t.set_postfix({'result':'update exist issue:'+name})
        else:
            issue_list.append({'project': {'id': 10000},
                               'issuetype':{'id':10100},
        'summary': name,
        'description': intro,
        'customfield_10201':code})
            issue=jira.create_issue(project='10000',issuetype='公司研究',summary=name,customfield_10201=code)
            issue.update(description=intro)
            #print('noissue:',name)

            t.set_postfix({'result':'noissue:'+name})

    

here:   5%|▍         | 14/296 [00:11<03:28,  1.36it/s, result=update exist issue:宝明科技]

error on : 宝明科技 002992.SZ


here:   5%|▌         | 15/296 [00:12<03:11,  1.47it/s, result=update exist issue:东尼电子]

error on : 东尼电子 603595.SH


here:  24%|██▎       | 70/296 [01:55<05:14,  1.39s/it, result=update exist issue:宝明科技]  

...

here:  95%|█████████▍| 280/296 [06:55<00:16,  1.00s/it, result=update exist issue:铖昌科技] 

error on : 铖昌科技 001270.SZ


here:  96%|█████████▌| 284/296 [07:00<00:14,  1.17s/it, result=update exist issue:横店影视]

error on : 横店影视 603103.SH


here: 100%|██████████| 296/296 [07:30<00:00,  1.52s/it, result=update exist issue:苏常柴Ａ]


In [101]:
jira._fields

{'cf[11441]': 'customfield_11441',
 'X值走廊参数': 'customfield_11441',
 'fixVersion': 'fixVersions',
 'cf[11440]': 'customfield_11440',
 '发生地域': 'customfield_11440',
 'cf[11443]': 'customfield_11443',
 '研究经验值': 'customfield_11443',
 'cf[11200]': 'customfield_11200',
 '组内停留时间': 'customfield_11200',
 'resolution': 'resolution',
 'cf[11442]': 'customfield_11442',
 '拟分配人员': 'customfield_11442',
 'cf[11201]': 'customfield_11201',
 '整体报告地址-废弃': 'customfield_11201',
 'cf[11720]': 'customfield_11720',
 '持仓价值': 'customfield_11720',
 'cf[11202]': 'customfield_11202',
 '资料包地址': 'customfield_11202',
 'cf[11203]': 'customfield_11203',
 '紧迫性等级': 'customfield_11203',
 'cf[11711]': 'customfield_11711',
 'QG_Code': 'customfield_11711',
 'cf[10621]': 'customfield_10621',
 '自动更新的近期市值均值': 'customfield_10621',
 'cf[10104]': 'customfield_10104',
 'Sprint': 'customfield_10104',
 'cf[11436]': 'customfield_11436',
 '推荐理由': 'customfield_11436',
 'cf[11710]': 'customfield_11710',
 'C指标操作台': 'customfield_11710',
 'cf

In [107]:
issue_list

[]

In [76]:
len(issue_list)

37

In [77]:
issues = jira.create_issues(field_list=issue_list)

In [5]:
issue=jira.issue(id='COMPSTUDY-5679')
issue

<JIRA Issue: key='COMPSTUDY-5679', id='27975'>

In [7]:
issue.fields.__dict__.get('customfield_11216','None')

'公司在空心杯电机，无框力矩电机，力传感器，谐波减速器方面均有产品，国内目前只有两家公司能做无框力矩电机，一家是步科股份，另一家为昊志机电；公司掌握六维力矩传感器技术，一个六维力矩传感器的价值量是2万元；减速器年产能为15万台。'

In [20]:
issue.fields.summary,issue.fields.description

('昊志机电', None)

In [18]:
issue

NameError: name 'issue' is not defined

In [103]:
issue.update(customfield_11216='A new summary was added')

In [41]:
issue=issue[0]
issue

<JIRA Issue: key='COMPSTUDY-5679', id='27975'>

In [46]:
issue.fields.__dict__

{'customfield_11441': None,
 'customfield_11200': None,
 'customfield_11443': 0.0,
 'resolution': <JIRA Resolution: name='完成', id='10101'>,
 'customfield_11201': None,
 'customfield_11442': None,
 'customfield_11202': None,
 'customfield_11720': 0.0,
 'customfield_11203': '0|i02edj:',
 'customfield_10621': None,
 'customfield_10104': None,
 'customfield_11436': None,
 'customfield_10622': None,
 'customfield_10105': '1|i02fjj:',
 'customfield_10623': None,
 'customfield_11438': None,
 'customfield_10625': None,
 'customfield_10900': None,
 'customfield_10626': '尚无预测',
 'customfield_11439': None,
 'customfield_10903': '尚未纳入',
 'customfield_11716': None,
 'customfield_10508': '未知',
 'customfield_10904': None,
 'customfield_10509': None,
 'customfield_10905': None,
 'lastViewed': '2023-07-10T18:55:50.925+0800',
 'priority': <JIRA Priority: name='跟随', id='10301'>,
 'customfield_10100': None,
 'customfield_10620': '2022-11-01',
 'customfield_10610': '狂热',
 'customfield_11303': None,
 'custo

In [45]:
issue.update(fields={'customfield_10201':'300503.sz'})

In [16]:
issue

NameError: name 'issue' is not defined

Reverse：

从codelist 获取jira和行情信息，整理成 各个字段+最终文本的xls表

In [104]:
import pandas as pd
import sys
from jira import JIRA
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL,get_mysql
from datetime import date

In [4]:
'601919.SH,600519.sh'.split(',')

['601919.SH', '600519.sh']

In [4]:
codelist=['601919.SH','600519.sh','abcd.qg']

In [5]:
class ScoutHelper():
    def __init__(self,codelist) -> None:
        self.codelist=[code.upper() for code in codelist]
        self.__jira=None
        self.__mysql=None
        self.no_data_list=[]
        self.no_jira_list=[]
        pass
    @property
    def mysql(self):

        if not hasattr(self, '__mysql') or self.__mysql is None:
                # 连接到MySQL
                self.__mysql = get_mysql('qgdbs')
                return self.__mysql
        
    @property
    def jira(self):

        if not hasattr(self, '__jira') or self.__jira is None:
                # 连接到MySQL
                self.__jira = JIRA('https://research.quantumgalaxy.cn/', basic_auth=('bot2', "jira_bot2")) 
                return self.__jira
        
            

    def fecth_ticker_data(self):
        #市值、涨幅
        #sql=f'select a.code,a.date,a.market_value,change_rate_1d from ticker_data a join processed_data b on a.code=b.code where a.date=b.date and a.code={code} order by date desc limit 1'
        sql=f'''SELECT a.code, a.date, a.market_value, b.change_rate_1d\
            FROM ticker_data a\
            JOIN processed_data b ON a.code = b.code AND a.date = b.date\
            WHERE a.code IN {tuple(self.codelist)}\
            AND (\
            SELECT COUNT(*)\
            FROM ticker_data\
            WHERE code = a.code AND date > a.date\
            ) = 0'''
        ticker_data=self.mysql.read_query(sql)
        self.mysql_df=pd.DataFrame(data=ticker_data,columns=['code','date','market_value','change_1d'])
        self.mysql_df=self.mysql_df.round({'market_value':2})
        self.mysql_df['change_1d']=self.mysql_df['change_1d'].apply(lambda x: '{:.2%}'.format(x))
        tickers=[t[0] for t in ticker_data]
        for ticker in self.codelist:
            if ticker not in tickers:
                self.no_data_list.append(ticker)
        return self.mysql_df

    def fecth_jira_data(self):
        #summary,description  侦察逻辑customfield_11216
        jira_data=[]
        for code in self.codelist:
            get_issue=self.jira.search_issues(f'project = COMPSTUDY AND 万得代码 ={code}')
            if not get_issue:
                self.no_jira_list.append(code)
            else:
                issue=get_issue[0]
                summary,description=issue.fields.summary,issue.fields.description
                jira_data.append((code,summary,description))
        self.jira_df=pd.DataFrame(data=jira_data,columns=['code','summary','description'])
    
    def work(self):
        self.fecth_jira_data()
        self.fecth_ticker_data()
        df=pd.merge(self.jira_df,self.mysql_df,on='code')
        if self.no_data_list:
             print('no_data_list:',self.no_data_list)
        if self.no_jira_list:
             print('no_jira_list',self.no_jira_list)
        if not (self.no_data_list and self.no_jira_list):
             print('all good')
        df['text']=None
        for i,r in df.iterrows():
            df.loc[i,'text']=f"""{r['summary']}{r['code']}({r['market_value']}亿元,{r['change_1d']})：{r['description']}"""
        
        return df
    def main(self):
        df=self.work()
        df1=pd.DataFrame(data=[['这些标的没有行情信息：',self.no_data_list],['这些标的没有jira问题：',self.no_jira_list]])
        #shape=df.shape[0]
        path=r'E:\wangzhilin\QuantumGalaxy\opt\scout\files'+f'\{date.today()}.xlsx'
        with pd.ExcelWriter(path) as writer:
            df.to_excel(writer, sheet_name='Sheet1', index=False)
            df1.to_excel(writer,sheet_name='Sheet2',index=False)
        #worksheet.write(f'A{shape+1}', '这些标的没有行情信息：'+self.no_data_list)
        #worksheet.write(f'A{shape+2}', '这些标的没有jira问题：'+self.no_jira_list)
        print(df1)
        return df
    def __del__(self):
        if self.__mysql is not None:
            self.__mysql.close()
        

In [2]:
r=''.split(',')
r

['']

In [4]:
if len(r)<=1:
    print('n')

n


In [139]:
scout=ScoutHelper(codelist)
scout

In [140]:
df=scout.main()

no_data_list: ['ABCD.QG']
no_jira_list ['ABCD.QG']
               0          1
0    这些标的没有行情信息：  [ABCD.QG]
1  这些标的没有jira问题：  [ABCD.QG]


In [136]:
scout.no_data_list

['ABCD.QG']

In [8]:
from datetime import datetime
datetime.now().strftime("%Y-%m-%d %H:%M:%S %f")

'2023-07-14 00:50:00 962576'

In [13]:
df=pd.DataFrame([1,2,3])
df

,0
0,1
1,2
2,3


In [16]:
import os

In [22]:

with pd.ExcelWriter(path) as writer:
    
    df.to_excel(writer, sheet_name='Sheet1', index=False)
    pass


In [10]:
r'E:\wangzhilin\QuantumGalaxy\opt\scout\files'+f'\{datetime.now().strftime("%Y-%m-%d_%H:%M:%S_%f")}.xlsx'

'E:\\wangzhilin\\QuantumGalaxy\\opt\\scout\\files\\2023-07-14_00:51:03_785883.xlsx'

In [7]:
datetime.now().strftime()

TypeError: strftime() missing required argument 'format' (pos 1)

In [103]:
f'E:\wangzhilin\QuantumGalaxy\opt\scout\files\{date.today()}.xlsx'

'E:\\wangzhilin\\QuantumGalaxy\\opt\\scout\x0ciles\\2023-07-13.xlsx'

In [64]:
df.to_excel(r'E:\wangzhilin\QuantumGalaxy\opt\scout\files\sample.xlsx')

In [77]:
mysql_df.round({'market_value':2})
mysql_df['change_1d']=mysql_df['change_1d'].apply(lambda x: '{:.2%}'.format(x))
mysql_df

,code,date,market_value,change_1d
0,600519.SH,2023-07-12,21528.717896,0.22%
1,601919.SH,2023-07-12,1487.218763,-1.32%


In [30]:
jira_data=[]
for code in codelist:
    get_issue=jira.search_issues(f'project = COMPSTUDY AND 万得代码 ={code}')
    if not get_issue:
        pass
    else:
        issue=get_issue[0]
        summary,description=issue.fields.summary,issue.fields.description
        jira_data.append((code,summary,description))
jira_data

[('601919.SH', '中远海控', '需求向好，格局向好，公司正在价值重估道路上。'), ('600519.SH', '贵州茅台', None)]

In [34]:
mysql_df=pd.DataFrame(data=ticker_data,columns=['code','date','market_value','change_1d'])

In [40]:
jira_df=pd.DataFrame(data=jira_data,columns=['code','name','d'])


In [41]:
df1=pd.merge(jira_df,mysql_df,on='code')
df1

,code,name,d,date,market_value,change_1d
0,601919.SH,中远海控,需求向好，格局向好，公司正在价值重估道路上。,2023-07-12,1487.218763,-0.013171
1,600519.SH,贵州茅台,None,2023-07-12,21528.717896,0.002222


"SELECT a.code, a.date, a.market_value, b.change_rate_1d            FROM ticker_data a            JOIN processed_data b ON a.code = b.code AND a.date = b.date            WHERE a.code IN ('601919.SH', '600519.sh')            AND (            SELECT COUNT(*)            FROM ticker_data            WHERE code = a.code AND date > a.date            ) = 0"

In [3]:
tuple(codelist)

NameError: name 'codelist' is not defined

In [6]:
scout=ScoutHelper(None)
scout.codelist

TypeError: 'NoneType' object is not iterable